In [2]:
#Imports
import open3d as o3d
from scipy.spatial import KDTree
import time
import random
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
mesh=o3d.io.read_triangle_mesh("plane_segments\Hyper_meshed_noise.stl")
mesh.merge_close_vertices(1e-5)
mesh.compute_vertex_normals()
points=np.asarray(mesh.vertices)
triangles=mesh.triangles
seed_point=random.choice(points)
print(seed_point)


[148.20359802 367.04119873  -0.5647074 ]


## Test 1 - KD Trees efficiency

In [15]:
############# Trial1 ###############
kd_tree=KDTree(points)
# Define bounds for the range search
bounds = [(seed_point[0]-20, seed_point[0]+20), (seed_point[1]-20, seed_point[1]+20), (seed_point[2]-20, seed_point[2]+20)]  # x in [1, 5], y in [2, 6], z in [3, 7]
s1=time.time()
# Filter points within the bounds
def range_search_kdtree(kd_tree, bounds):
    # Approximate bounds for the center point and a large enough radius
    center = [(b[0] + b[1]) / 2 for b in bounds]
    radius = np.linalg.norm([(b[1] - b[0]) / 2 for b in bounds])
    # Perform a broad search using query_ball_point
    candidate_indices = kd_tree.query_ball_point(center, radius)

    # Filter candidates based on bounds
    result = []
    for idx in candidate_indices:
        point = points[idx]
        if all(bounds[i][0] <= point[i] <= bounds[i][1] for i in range(len(bounds))):
            result.append(point)

    return result

# Perform the range search
results = range_search_kdtree(kd_tree, bounds)
e1=time.time()
print(f"Search time: {e1-s1}")
print("Points within bounds:", len(results))

Search time: 0.007323503494262695
Points within bounds: 729


In [16]:
########### Trial2 ################
s2=time.time()
results2=[]
for point in points:
    if (
        seed_point[0]-20 <= point[0] <= seed_point[0]+20 and
        seed_point[1]-20 <= point[1] <= seed_point[1]+20 and
        seed_point[2]-20 <= point[2] <= seed_point[2]+20
    ):
        results2.append(point)
e2=time.time()
print(f"Search time: {e2-s2}")
print("Points within bounds:", len(results2))

Search time: 0.05357789993286133
Points within bounds: 729


## Test 2 - Repeated KD-tree search

In [4]:
# Filter points within the bounds
def range_search_kdtree(kd_tree, bounds):
    # Approximate bounds for the center point and a large enough radius
    center = [(b[0] + b[1]) / 2 for b in bounds]
    radius = np.linalg.norm([(b[1] - b[0]) / 2 for b in bounds])
    # Perform a broad search using query_ball_point
    candidate_indices = kd_tree.query_ball_point(center, radius)

    # Filter candidates based on bounds
    result = []
    for idx in candidate_indices:
        point = points[idx]
        if all(bounds[i][0] <= point[i] <= bounds[i][1] for i in range(len(bounds))):
            result.append(point)

    return result

kd_tree=KDTree(points)
s1=time.time()
triangle_hits=[]
for i in range(4):
    for j in range(100):
        seed_point=random.choice(points)
        bounds = [(seed_point[0]-20, seed_point[0]+20), (seed_point[1]-20, seed_point[1]+20), (seed_point[2]-20, seed_point[2]+20)] 
        results = range_search_kdtree(kd_tree, bounds)
        for triangle_index, triangle in enumerate(triangles):
            tri_vertices = [points[vertex_index] for vertex_index in triangle]
            # Check if any vertex of the triangle falls within the prism
            if np.any(tuple(vertex) in results for vertex in tri_vertices):
                            triangle_hits.append(triangle_index)  # Add to set
                            break
e1=time.time()
print(f"Search time: {e1-s1}")
print("Triangle within bounds:", len(triangle_hits))



Search time: 0.9834747314453125
Triangle within bounds: 400


### Test 3 - KD trees and coordinate transform

In [5]:
num_vertices = points.shape[0]
colors = np.full((num_vertices, 3), 0.7)  # Light gray
kd_tree=KDTree(points)

s1=time.time()
for trial in range(400):

    seed_point=random.choice(points) 
    candidate_indices = np.array(kd_tree.query_ball_point(seed_point, 32))

    mask = np.all(colors[candidate_indices] == [0,0,1], axis=1)
    colors[candidate_indices[mask]] = [0, 1, 0]  # Red

    '''
    #Calculate "probe" in xyz orientation
    for index, i in enumerate(points[candidate_indices]):
        if (seed_point[0]-10<=i[0]<=seed_point[0]+10 and
            seed_point[1]-32<=i[1]<=seed_point[1]+32):
            colors[candidate_indices[index]]=[0,1,0]
    '''

    #Calculate "probe" rotated 45 degrees
    rotation_matrix=np.array([[np.cos(np.pi/2),   -np.sin(np.pi/2), 0],
                            [np.sin(np.pi/2),  np.cos(np.pi/2),  0],
                            [ 0, 0,  1 ]])
    inv_rotation_matrix = np.linalg.inv(rotation_matrix)

    new_candidate_points= inv_rotation_matrix@points[candidate_indices].T
    new_seed_point=inv_rotation_matrix@seed_point
    probe_mask = ((new_seed_point[0] - 10 <= new_candidate_points[0]) & (new_candidate_points[0] <= new_seed_point[0] + 10) &
                  (new_seed_point[1] - 32 <= new_candidate_points[1]) & (new_candidate_points[1] <= new_seed_point[1] + 32))
    
    colors[candidate_indices[probe_mask]] = [0, 0, 1]

    

e1=time.time()

# Assign colors to mesh
mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([mesh])
print(f"Total time: {e1-s1}")


Total time: 0.22311782836914062


### Test 4 - KD + coordinate transform with double-scanned color mapping

In [8]:
num_vertices = points.shape[0]
colors = np.full((num_vertices, 3), (1,0,0))
kd_tree=KDTree(points)

s1=time.time()
for trial in range(4):
    color_updates = np.full((num_vertices, 3), -1.0)  # Use -1 to mark unchanged
    for j in range(100):
        seed_point=random.choice(points) 
        candidate_indices = np.array(kd_tree.query_ball_point(seed_point, 32))

        mask = np.all(colors[candidate_indices] == [0,0,1], axis=1)
        color_updates[candidate_indices[mask]] = [0, 1, 0]  # Red

        '''
        #Calculate "probe" in xyz orientation
        for index, i in enumerate(points[candidate_indices]):
            if (seed_point[0]-10<=i[0]<=seed_point[0]+10 and
                seed_point[1]-32<=i[1]<=seed_point[1]+32):
                colors[candidate_indices[index]]=[0,1,0]
        '''

        #Calculate "probe" rotated 45 degrees
        rotation_matrix=np.array([[np.cos(np.pi/2),   -np.sin(np.pi/2), 0],
                                [np.sin(np.pi/2),  np.cos(np.pi/2),  0],
                                [ 0, 0,  1 ]])
        inv_rotation_matrix = np.linalg.inv(rotation_matrix)

        new_candidate_points= inv_rotation_matrix@points[candidate_indices].T
        new_seed_point=inv_rotation_matrix@seed_point
        probe_mask = ((new_seed_point[0] - 10 <= new_candidate_points[0]) & (new_candidate_points[0] <= new_seed_point[0] + 10) &
                    (new_seed_point[1] - 32 <= new_candidate_points[1]) & (new_candidate_points[1] <= new_seed_point[1] + 32))
        
        color_updates[candidate_indices[probe_mask]] = [0, 1, 0]  # Blue
    
    update_mask = color_updates[:, 0] != -1
    already_marked_mask = (colors[:, 0] != 1) 
    rescanned_mask = update_mask & already_marked_mask
    colors[update_mask] = color_updates[update_mask]
    colors[rescanned_mask]=[0,0,1]

e1=time.time()

# Assign colors to mesh
mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([mesh])
print(f"Total time: {e1-s1}")

Total time: 0.23128318786621094


### Test 5 - KD + coordinate transform with multi-scanned color mapping

In [11]:
colors = np.full((num_vertices, 3), (1, 0, 0))  # Red for unscanned
scan_counts = np.zeros(num_vertices, dtype=int)  # Track scan occurrences
kd_tree = KDTree(points)

# Color map for up to 10 scans (adjust as needed)
color_map = [
    (1, 0, 0),  # 0 scans  -> Red
    (0, 1, 0),  # 1 scan   -> Green
    (0, 0, 1),  # 2 scans  -> Blue
    (1, 1, 0),  # 3 scans  -> Yellow
    (1, 0, 1),  # 4 scans  -> Purple
    (0, 1, 1),  # 5 scans  -> Cyan
    (0.5, 0, 0),  # 6 scans  -> Dark Red
    (0, 0.5, 0),  # 7 scans  -> Dark Green
    (0, 0, 0.5),  # 8 scans  -> Dark Blue
    (0.5, 0.5, 0.5),  # 9+ scans -> Gray
]

s1 = time.time()
for trial in range(4):
    color_updates = np.full((num_vertices, 3), -1.0)  # Use -1 to mark unchanged
    scan_updates = np.zeros(num_vertices, dtype=int)  # Track scan increments
    
    for j in range(100):
        seed_point = random.choice(points)
        candidate_indices = np.array(kd_tree.query_ball_point(seed_point, 32))

        # Track redundant scans (Blue)
        mask = np.all(colors[candidate_indices] == [0, 0, 1], axis=1)
        scan_updates[candidate_indices[mask]] += 1  # Increment scan count

        # Calculate "probe" rotated 45 degrees
        rotation_matrix = np.array([
            [np.cos(np.pi/2), -np.sin(np.pi/2), 0],
            [np.sin(np.pi/2),  np.cos(np.pi/2), 0],
            [0, 0, 1]
        ])
        inv_rotation_matrix = np.linalg.inv(rotation_matrix)

        new_candidate_points = inv_rotation_matrix @ points[candidate_indices].T
        new_seed_point = inv_rotation_matrix @ seed_point
        probe_mask = (
            (new_seed_point[0] - 10 <= new_candidate_points[0]) & (new_candidate_points[0] <= new_seed_point[0] + 10) &
            (new_seed_point[1] - 32 <= new_candidate_points[1]) & (new_candidate_points[1] <= new_seed_point[1] + 32)
        )
        
        scan_updates[candidate_indices[probe_mask]] += 1  # Increment scan count

    # Apply scan count updates
    scan_counts += scan_updates

    # Assign new colors based on scan counts
    unique_scan_counts = np.clip(scan_counts, 0, len(color_map) - 1)  # Limit max color index
    colors = np.array([color_map[i] for i in unique_scan_counts])

e1 = time.time()

# Assign colors to mesh
mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([mesh])
print(f"Total time: {e1-s1}")

Total time: 0.33519959449768066
